# Coverage extension — 5 priority boards (headless GPU)

Solves ONLY the 5 new boards (indices 12-16: the highest-frequency textures the priority scorer flagged as uncovered). ~2.5 h. You already have boards 0-11 in `flop_pack_v1_fullrange.db`, so this does NOT re-solve them.

1. Settings -> **GPU T4 x2 (or P100)** + **Internet On**.
2. **Save & Run All (Commit)**, close the tab.
3. Output tab -> download **`records_ext.json`** and send it back — it gets merged + signed with the existing pack locally.

In [ ]:
# Fail fast if no GPU.
import subprocess
try:
    _r = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True)
except FileNotFoundError:
    raise SystemExit('No nvidia-smi (CPU instance) -- set Accelerator -> GPU, then re-run.')
assert _r.returncode == 0 and 'GPU' in _r.stdout, 'NO GPU -- set Accelerator -> GPU T4 x2, then re-run.'
print(_r.stdout.strip())

In [ ]:
# Clone the solver source from the public repo (needs Internet On); pulls the 5 new boards.
!rm -rf /kaggle/working/poker && git clone -q --depth 1 https://github.com/tian-chaiyaporn2/poker_offline_trainer /kaggle/working/poker
import os
from importlib import util
import sys; sys.path.insert(0, '/kaggle/working/poker/src')
from pokertrainer.presets import BOARDS
print('boards available:', len(BOARDS), '| extension (12-16):', [b['board'] for b in BOARDS[12:]])

In [ ]:
try:
    import cupy as cp
except Exception:
    import subprocess,sys; subprocess.run([sys.executable,'-m','pip','install','-q','cupy-cuda12x']); import cupy as cp
free,total=cp.cuda.Device(0).mem_info
nm=cp.cuda.runtime.getDeviceProperties(0)['name']; print('cupy',cp.__version__,'|',nm.decode() if isinstance(nm,bytes) else nm)
print('GPU memory: %.1f GB free / %.1f GB total'%(free/1e9,total/1e9))

## Solve the 5 new boards (~2.5 h, checkpointed)

In [ ]:
!cd /kaggle/working/poker && PYTHONPATH=src python -m pokertrainer.content_yield \
    --solver gpu --dtype float32 --n 400 --iters 300 --roots 12,13,14,15,16 --out /kaggle/working/cy_ext

In [ ]:
# Expose records_ext.json for download (fails loudly if the run produced nothing).
import json, shutil, os
try:
    rep = json.load(open('/kaggle/working/cy_ext/yield_report.json'))
except (FileNotFoundError, json.JSONDecodeError):
    raise SystemExit('No cy_ext output -- check the cell above and /kaggle/working/cy_ext/boards/*.ERROR.txt')
done, missing = rep.get('boards_completed') or [], rep.get('boards_missing') or []
print('boards completed:', done, '| missing:', missing)
if not done:
    raise SystemExit('No boards completed -- nothing to download. Check cy_ext/boards/*.ERROR.txt')
shutil.copy('/kaggle/working/cy_ext/records.json', '/kaggle/working/records_ext.json')
print('DOWNLOAD -> records_ext.json (%d KB, %d records across boards %s)'
      % (os.path.getsize('/kaggle/working/records_ext.json')//1024, len(json.load(open('/kaggle/working/cy_ext/records.json'))), done))